# 03 - Modeling

Rubric: *Modeling (5)*. Four model families - DecisionTree and RandomForest (lectures), Logistic regression and
XGBoost (new) - predict `is_alpha_5` for liquid Thai stocks.

* **Splits:** train < 2020, validation 2020-2021, test 2022 onward, 5-day embargo at each boundary.
* **Tuning:** grid search on the validation years; the selection metric is the 5-day alpha of each day's
  top-10% picks (what the strategy trades), with AUC reported alongside.
* **Walk-forward:** test years are predicted by models refit every January on all earlier data.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from setalpha import config, ingest, transform, model, regime, portfolio, strategy, evaluate, report, pipeline
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

In [ ]:
logs = pd.read_csv(config.REPORTS_DIR / "tuning_log.csv")
logs.sort_values(["model", "valid_top_decile_alpha_%"], ascending=[True, False]).groupby("model").head(3)

In [ ]:
meta = report.load_meta()
print("best params:", meta["best_params"]); print("validation IC of the ensemble:", round(meta["ic_valid"], 4))
print("custom decision thresholds (95th pct of validation probabilities):", meta["thr"])

In [ ]:
df = transform.load()
preds = pd.read_parquet(model.PREDICTIONS)
full = preds.merge(df[["Date", "Ticker", "tradable", "fwd_alpha_5", "is_alpha_5"]], on=["Date", "Ticker"])
scores = [f"pred_{n}" for n in model.MODEL_NAMES] + ["pred_ens"]
for sp in ["valid", "test"]:
    print(f"\n{sp.upper()}"); display(model.scorecard(full[full["split"] == sp], scores).round(4))

## Custom decision rules

The trading rules act on the *highest-probability* events only: a probability above the validation 95th
percentile, the daily top-5 of the RF/XGB ensemble, or both models agreeing. Precision rises as the rule gets
stricter:

In [ ]:
test = full[(full["split"] == "test") & full["tradable"]]
rows = {}
for q in [0.5, 0.8, 0.9, 0.95, 0.98]:
    thr = full.loc[(full["split"] == "valid") & full["tradable"], "pred_xgb"].quantile(q)
    sel = test[test["pred_xgb"] >= thr]
    rows[f"XGB prob >= valid q{int(q*100)}"] = {"picks": len(sel), "precision %": 100 * sel["is_alpha_5"].mean(),
                                                "avg 5d alpha %": 100 * sel["fwd_alpha_5"].mean()}
pd.DataFrame(rows).T

In [ ]:
ic = {s: model.daily_ic(full[full["split"] == "test"], s) for s in ["pred_xgb", "pred_rf", "pred_ens"]}
ic_df = pd.DataFrame(ic)
ic_df.groupby(ic_df.index.year).mean().plot.bar(title="Mean daily IC by test year", figsize=(10, 3));
ic_df.rolling(60).mean().plot(title="60-day rolling IC", figsize=(12, 3));

In [ ]:
xgb = model.load_final("xgb")
model.feature_importance(xgb["model"], xgb["features"]).sort_values().plot.barh(figsize=(8, 8), title="XGBoost importance (production model)");